# Notebook 03 -- Arquitectura SCE-Net en PyTorch

Implementacion completa del modelo **Similarity Condition Embedding Network (SCE-Net)** con:
- Backbone **ResNet18** preentrenado en ImageNet y fine-tunable end-to-end
- **M=5 condition masks** aprendibles de dimension D=64
- **Condition weight branch** multimodal (fusion Visual x Texto Fisher HGLMM)
- **Triplet loss** con regularizacion L1 (masks) + L2 (embeddings)

Referencia: *Tan et al., Learning Similarity Conditions Without Explicit Supervision, 2019*

## Configuracion

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json

# Rutas
OUT_DIR   = Path('./polyvore_kaggle/polyvore_outfits/data/processed')
DATA_ROOT = Path('./polyvore_kaggle')

# Hiperparametros del modelo (fieles al paper)
EMBED_DIM    = 64     # dimension embedding visual (paper: 64)
M_CONDITIONS = 5      # numero de condition masks (paper: 5 para Polyvore)
TEXT_DIM     = 2048   # dimension Fisher Vector tras PCA
IMG_SIZE     = 224    # input size ResNet18
MARGIN       = 0.3    # margen triplet loss
LAMBDA_L1    = 5e-4   # regularizacion L1 sobre masks
LAMBDA_L2    = 5e-4   # regularizacion L2 sobre embeddings

# Dispositivo
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Dispositivo     : {DEVICE}')
print(f'\nEMBED_DIM    : {EMBED_DIM}')
print(f'M_CONDITIONS : {M_CONDITIONS}')
print(f'TEXT_DIM     : {TEXT_DIM}')


## 1. ResNet18 backbone

Usamos `torchvision.models.resnet18` preentrenado en ImageNet, que es exactamente lo que usa el paper.

Modificaciones respecto al ResNet18 original:
- Sustituimos la cabeza de clasificacion (fc de 1000 clases) por una proyeccion lineal a `EMBED_DIM=64`
- El backbone completo es **fine-tunable end-to-end** (ninguna capa congelada)

Esto replica fielmente la seccion 4.2 del paper:
> *'We use an 18-layer deep residual network pretrained on ImageNet and fine-tuned during training'*

In [ ]:
import torchvision.models as models

class ResNet18Backbone(nn.Module):
    """
    ResNet18 preentrenado en ImageNet con cabeza sustituida.
    Proyecta imagenes (B, 3, 224, 224) a embeddings (B, embed_dim).
    Todos los parametros son entrenables (fine-tuning end-to-end).
    """
    def __init__(self, embed_dim):
        super().__init__()

        # Cargar ResNet18 preentrenado en ImageNet
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Extraer todo menos la cabeza de clasificacion original
        self.features = nn.Sequential(*list(base.children())[:-1])
        # children()[:-1] = conv1, bn1, relu, maxpool, layer1-4, avgpool
        # Output: (B, 512, 1, 1)

        # Cabeza de proyeccion: 512 -> embed_dim
        self.projection = nn.Linear(512, embed_dim, bias=False)

    def forward(self, x):
        """
        Args:
            x: (B, 3, H, W) imagen normalizada
        Returns:
            v: (B, embed_dim) embedding visual L2-normalizado
        """
        feat = self.features(x)          # (B, 512, 1, 1)
        feat = feat.flatten(1)           # (B, 512)
        v    = self.projection(feat)     # (B, embed_dim)
        v    = F.normalize(v, p=2, dim=-1)  # L2 normalizacion
        return v


# Instanciar y verificar
backbone = ResNet18Backbone(EMBED_DIM).to(DEVICE)

dummy_img = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = backbone(dummy_img)

print(f'Input  shape: {dummy_img.shape}')
print(f'Output shape: {out.shape}  (esperado: (2, {EMBED_DIM}))')
print(f'Norma L2 (debe ser ~1.0): {out.norm(dim=-1)}')

n_params = sum(p.numel() for p in backbone.parameters())
print(f'Parametros ResNet18: {n_params:,}')


## 2. Condition Masks

M=5 mascaras aprendibles de dimension D=64.

Aplicacion (ecuacion 1 del paper):
```
E_ij = C_j * V_i   (producto elemento a elemento)
```

Regularizacion L1 para forzar **sparsity y desacoplamiento** entre mascaras (ecuacion 5).

In [ ]:
class ConditionMasks(nn.Module):
    """
    M mascaras de condicion aprendibles de dimension D.
    Parametros: masks (M, D)

    Aplicacion vectorizada sobre un batch:
      O = V.unsqueeze(1) * masks.unsqueeze(0)  -> (B, M, D)
    """
    def __init__(self, m_conditions, embed_dim):
        super().__init__()
        self.m_conditions = m_conditions
        self.embed_dim    = embed_dim

        # Inicializacion uniforme en [0,1] -- mascaras positivas al inicio
        self.masks = nn.Parameter(
            torch.rand(m_conditions, embed_dim)
        )

    def forward(self, v):
        """
        Args:
            v: (B, D) embedding visual
        Returns:
            O: (B, M, D) embeddings enmascarados para cada condicion
        """
        # v:     (B, D)    -> (B, 1, D)
        # masks: (M, D)    -> (1, M, D)
        return v.unsqueeze(1) * self.masks.unsqueeze(0)   # (B, M, D)

    def l1_loss(self):
        """Regularizacion L1 sobre las mascaras (paper ecuacion 5)."""
        return self.masks.abs().sum()


# Test
cond_masks = ConditionMasks(M_CONDITIONS, EMBED_DIM).to(DEVICE)
dummy_v    = torch.zeros(4, EMBED_DIM).to(DEVICE)
O          = cond_masks(dummy_v)
print(f'Output ConditionMasks: {O.shape}  (esperado: (4, {M_CONDITIONS}, {EMBED_DIM}))')
print(f'Parametros masks: {cond_masks.masks.numel():,}')


## 3. Condition Weight Branch

MLP que recibe el par de items y produce un vector de pesos w de dimension M.

Variante **Visual-Text** del paper (seccion 3.3, ecuacion 7):
```
vt_i = V_i * T_i_proj   (producto elemento a elemento)
vt_j = V_j * T_j_proj
y    = concat(vt_i, vt_j)  -> MLP -> Softmax -> w (M,)
```

El texto se proyecta primero a `embed_dim` para poder multiplicarlo por el embedding visual.

In [ ]:
class ConditionWeightBranch(nn.Module):
    """
    Weight branch multimodal Visual-Text del paper SCE-Net (seccion 3.3).

    Fusion: producto elemento a elemento V * T_proj
    El paper reporta que es superior a concatenacion directa.

    Entrada : (V_i, V_j) embeddings visuales + (T_i, T_j) Fisher Vectors
    Salida  : w (B, M) pesos de condicion normalizados por softmax
    """
    def __init__(self, m_conditions, embed_dim, text_dim, hidden_dim=None):
        super().__init__()

        # Proyeccion texto -> embed_dim para fusion V*T
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, embed_dim),
            nn.ReLU(),
        )

        # MLP: concat(vt_i, vt_j) -> w
        # Input: 2 * embed_dim
        hidden = hidden_dim or max(2 * embed_dim, 64)
        self.mlp = nn.Sequential(
            nn.Linear(2 * embed_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, m_conditions),
        )
        # Softmax aplicado en forward para mayor control numerico

    def forward(self, v_i, v_j, t_i, t_j):
        """
        Args:
            v_i, v_j : (B, embed_dim)  embeddings visuales L2-normalizados
            t_i, t_j : (B, text_dim)   Fisher Vectors
        Returns:
            w: (B, M) pesos de condicion (suman 1 por softmax)
        """
        # Proyectar texto a embed_dim
        ti_proj = self.text_proj(t_i)   # (B, embed_dim)
        tj_proj = self.text_proj(t_j)   # (B, embed_dim)

        # Fusion visual-texto: producto elemento a elemento (paper ec. 7)
        vt_i = v_i * ti_proj            # (B, embed_dim)
        vt_j = v_j * tj_proj            # (B, embed_dim)

        # Concatenar y pasar por MLP
        y = torch.cat([vt_i, vt_j], dim=-1)   # (B, 2*embed_dim)
        w = F.softmax(self.mlp(y), dim=-1)    # (B, M)
        return w


# Test
weight_branch = ConditionWeightBranch(M_CONDITIONS, EMBED_DIM, TEXT_DIM).to(DEVICE)
v_i = torch.zeros(4, EMBED_DIM).to(DEVICE)
v_j = torch.zeros(4, EMBED_DIM).to(DEVICE)
t_i = torch.zeros(4, TEXT_DIM).to(DEVICE)
t_j = torch.zeros(4, TEXT_DIM).to(DEVICE)
w   = weight_branch(v_i, v_j, t_i, t_j)
print(f'Output WeightBranch : {w.shape}  (esperado: (4, {M_CONDITIONS}))')
print(f'Suma pesos (debe ser 1.0): {w.sum(dim=-1)}')


## 4. SCE-Net completo

Integra los tres componentes siguiendo la figura 2 y ecuaciones 1-2 del paper:

```
(img_i, img_j) -> ResNet18 -> (V_i, V_j)          embeddings visuales
(txt_i, txt_j) -> Fisher Vector FIJO -> (T_i, T_j)

(V_i,T_i),(V_j,T_j) -> WeightBranch -> w  (B, M)

V_i -> ConditionMasks -> O_i  (B, M, D)
V_j -> ConditionMasks -> O_j  (B, M, D)

# Ecuacion 2: E_i = w * O_i  (suma ponderada sobre M condiciones)
E_i = (w.unsqueeze(-1) * O_i).sum(dim=1)   (B, D)
E_j = (w.unsqueeze(-1) * O_j).sum(dim=1)   (B, D)
```

In [ ]:
class SCENet(nn.Module):
    """
    Similarity Condition Embedding Network -- Tan et al. 2019

    Modo multimodal Visual-Text:
    - ResNet18 fine-tunable end-to-end
    - M condition masks aprendibles
    - Weight branch con fusion V*T
    """
    def __init__(self, embed_dim, m_conditions, text_dim, img_size=224):
        super().__init__()
        self.embed_dim    = embed_dim
        self.m_conditions = m_conditions

        self.backbone      = ResNet18Backbone(embed_dim)
        self.cond_masks    = ConditionMasks(m_conditions, embed_dim)
        self.weight_branch = ConditionWeightBranch(
            m_conditions, embed_dim, text_dim)

    def encode(self, img):
        """
        Embedding visual en espacio general (antes de aplicar masks).
        Args:
            img: (B, 3, H, W)
        Returns:
            v: (B, embed_dim) L2-normalizado
        """
        return self.backbone(img)

    def get_final_embedding(self, v_i, v_j, t_i, t_j):
        """
        Calcula embeddings finales E_i, E_j para un par.

        Ecuacion 2 del paper:
          w   = WeightBranch(V_i, V_j, T_i, T_j)  (B, M)
          O_i = ConditionMasks(V_i)                (B, M, D)
          E_i = sum_m w_m * O_im                   (B, D)

        Args:
            v_i, v_j: (B, embed_dim) embeddings visuales
            t_i, t_j: (B, text_dim)  Fisher Vectors
        Returns:
            E_i, E_j: (B, embed_dim) embeddings finales
        """
        # Pesos de condicion
        w = self.weight_branch(v_i, v_j, t_i, t_j)  # (B, M)

        # Embeddings enmascarados
        O_i = self.cond_masks(v_i)   # (B, M, D)
        O_j = self.cond_masks(v_j)   # (B, M, D)

        # Suma ponderada sobre las M condiciones (ecuacion 2)
        w_exp = w.unsqueeze(-1)                          # (B, M, 1)
        E_i   = (w_exp * O_i).sum(dim=1)                # (B, D)
        E_j   = (w_exp * O_j).sum(dim=1)                # (B, D)

        return E_i, E_j

    def forward(self, img_a, img_p, img_n, txt_a, txt_p, txt_n):
        """
        Forward pass para una tripleta (anchor, positive, negative).

        Args:
            img_a, img_p, img_n: (B, 3, H, W)  imagenes
            txt_a, txt_p, txt_n: (B, text_dim)  Fisher Vectors
        Returns:
            E_a, E_p, E_n: (B, embed_dim) embeddings finales
            v_a, v_p, v_n: (B, embed_dim) embeddings visuales generales
                           (necesarios para la regularizacion L2)
        """
        # Embeddings visuales generales
        v_a = self.encode(img_a)
        v_p = self.encode(img_p)
        v_n = self.encode(img_n)

        # Par (anchor, positive)
        E_a1, E_p = self.get_final_embedding(v_a, v_p, txt_a, txt_p)
        # Par (anchor, negative)
        E_a2, E_n = self.get_final_embedding(v_a, v_n, txt_a, txt_n)

        # Promedio de los dos embeddings del anchor
        E_a = (E_a1 + E_a2) / 2.0

        return E_a, E_p, E_n, v_a, v_p, v_n


# Instanciar
model = SCENet(
    embed_dim    = EMBED_DIM,
    m_conditions = M_CONDITIONS,
    text_dim     = TEXT_DIM,
    img_size     = IMG_SIZE,
).to(DEVICE)

# Test forward pass con datos dummy
B = 4
with torch.no_grad():
    E_a, E_p, E_n, v_a, v_p, v_n = model(
        torch.zeros(B, 3, IMG_SIZE, IMG_SIZE).to(DEVICE),
        torch.zeros(B, 3, IMG_SIZE, IMG_SIZE).to(DEVICE),
        torch.zeros(B, 3, IMG_SIZE, IMG_SIZE).to(DEVICE),
        torch.zeros(B, TEXT_DIM).to(DEVICE),
        torch.zeros(B, TEXT_DIM).to(DEVICE),
        torch.zeros(B, TEXT_DIM).to(DEVICE),
    )

print(f'E_a : {E_a.shape}  (esperado: ({B}, {EMBED_DIM}))')
print(f'E_p : {E_p.shape}')
print(f'E_n : {E_n.shape}')
print(f'v_a : {v_a.shape}')

total_params    = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.backbone.parameters())
other_params    = total_params - backbone_params
print(f'\nParametros totales         : {total_params:,}')
print(f'  ResNet18 backbone        : {backbone_params:,}')
print(f'  Masks + Weight Branch    : {other_params:,}')


## 5. Funciones de perdida

Siguiendo la ecuacion 5 del paper:

```
L_final = L_triplet + lambda1 * L1(masks) + lambda2 * L2(embeddings)
```

**Triplet loss** (ecuacion 4):
```
L_triplet = max(0, ||E_a - E_p|| - ||E_a - E_n|| + margin)
```

In [ ]:
def triplet_loss(E_a, E_p, E_n, margin=MARGIN):
    """
    Triplet loss con distancia euclidea (paper ecuacion 4).
    L = mean( max(0, d(E_a,E_p) - d(E_a,E_n) + margin) )
    """
    d_pos = (E_a - E_p).norm(dim=-1)   # (B,)
    d_neg = (E_a - E_n).norm(dim=-1)   # (B,)
    loss  = F.relu(d_pos - d_neg + margin)
    return loss.mean()


def l2_reg(v):
    """Regularizacion L2 sobre embeddings visuales."""
    return (v ** 2).sum(dim=-1).mean()


def total_loss(model, E_a, E_p, E_n, v_a, v_p, v_n,
               lambda_l1=LAMBDA_L1, lambda_l2=LAMBDA_L2, margin=MARGIN):
    """
    Loss total del paper (ecuacion 5):
      L = L_triplet + lambda1*L1(masks) + lambda2*L2(embeddings)

    Returns:
        loss      : loss total
        l_triplet : solo la triplet loss (para logging)
        l_l1      : regularizacion L1
        l_l2      : regularizacion L2
    """
    l_triplet = triplet_loss(E_a, E_p, E_n, margin)
    l_l1      = model.cond_masks.l1_loss()
    l_l2      = (l2_reg(v_a) + l2_reg(v_p) + l2_reg(v_n)) / 3.0

    loss = l_triplet + lambda_l1 * l_l1 + lambda_l2 * l_l2
    return loss, l_triplet, l_l1, l_l2


# Test con tensores aleatorios
E_a_t = torch.randn(4, EMBED_DIM)
E_p_t = torch.randn(4, EMBED_DIM)
E_n_t = torch.randn(4, EMBED_DIM)
l_t   = triplet_loss(E_a_t, E_p_t, E_n_t)
print(f'Triplet loss (test aleatorio): {l_t.item():.4f}')
print('Loss functions OK')


## 6. Preprocesamiento de imagenes

Pipeline compatible con ResNet18 preentrenado en ImageNet:
- Resize a 224x224
- Normalizacion con media y std de ImageNet
- Data augmentation durante entrenamiento (flip horizontal + random crop)

In [ ]:
from torchvision import transforms
from PIL import Image

# Transformaciones ImageNet estandar
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def load_image(img_path, transform):
    """
    Carga y preprocesa una imagen.
    Args:
        img_path: string o Path
        transform: torchvision transform
    Returns:
        tensor: (3, H, W) float32
    """
    img = Image.open(img_path).convert('RGB')
    return transform(img)


# Test con una imagen real del dataset
with open(OUT_DIR / 'item_index.json') as f:
    item_index = json.load(f)

sample_id   = next(iter(item_index))
sample_path = DATA_ROOT / item_index[sample_id]['image_path']
print(f'Imagen de prueba: {sample_path}')

img_test = load_image(sample_path, transform_val)
print(f'Shape : {img_test.shape}  (esperado: (3, {IMG_SIZE}, {IMG_SIZE}))')
print(f'Min   : {img_test.min():.3f}')
print(f'Max   : {img_test.max():.3f}')


## 7. Verificacion end-to-end con datos reales

Test completo con una tripleta real del dataset para confirmar que todo el pipeline funciona antes de entrenar.

In [ ]:
# Cargar Fisher Vectors
fisher_matrix = np.load(OUT_DIR / 'fisher_vectors.npy')
with open(OUT_DIR / 'fisher_item_ids.json') as f:
    fisher_ids = json.load(f)
fv_lookup = {iid: idx for idx, iid in enumerate(fisher_ids)}
print(f'Fisher Vectors: {fisher_matrix.shape}')

# Cargar una tripleta real
with open(OUT_DIR / 'triplets_train.json') as f:
    triplets = json.load(f)

t   = triplets[0]
ids = [t['anchor_id'], t['pos_id'], t['neg_id']]

imgs, txts = [], []
for iid in ids:
    img_path = DATA_ROOT / item_index[iid]['image_path']
    imgs.append(load_image(img_path, transform_val))
    fv = fisher_matrix[fv_lookup[iid]] if iid in fv_lookup \
         else np.zeros(TEXT_DIM, dtype=np.float32)
    txts.append(torch.tensor(fv, dtype=torch.float32))

# Crear batch de 1 y mover a GPU
img_a = imgs[0].unsqueeze(0).to(DEVICE)
img_p = imgs[1].unsqueeze(0).to(DEVICE)
img_n = imgs[2].unsqueeze(0).to(DEVICE)
txt_a = txts[0].unsqueeze(0).to(DEVICE)
txt_p = txts[1].unsqueeze(0).to(DEVICE)
txt_n = txts[2].unsqueeze(0).to(DEVICE)

# Forward pass
model.eval()
with torch.no_grad():
    E_a, E_p, E_n, v_a, v_p, v_n = model(
        img_a, img_p, img_n, txt_a, txt_p, txt_n)

d_pos = (E_a - E_p).norm().item()
d_neg = (E_a - E_n).norm().item()

print(f'Tripleta de prueba:')
print(f'  anchor : {t["anchor_id"]}  ({item_index[t["anchor_id"]]["category"]})')
print(f'  pos    : {t["pos_id"]}  ({item_index[t["pos_id"]]["category"]})')
print(f'  neg    : {t["neg_id"]}  ({item_index[t["neg_id"]]["category"]})')
print(f'\nDistancia anchor-positive : {d_pos:.4f}')
print(f'Distancia anchor-negative : {d_neg:.4f}')
print(f'(Sin entrenar ambas distancias son similares -- correcto)')

# Verificar loss
loss, l_t, l_l1, l_l2 = total_loss(model, E_a, E_p, E_n, v_a, v_p, v_n)
print(f'\nLoss total    : {loss.item():.4f}')
print(f'  L_triplet   : {l_t.item():.4f}')
print(f'  L1 (masks)  : {l_l1.item():.4f}')
print(f'  L2 (embed)  : {l_l2.item():.4f}')

print('\nVerificacion end-to-end OK')


## 8. Guardar arquitectura y configuracion

In [ ]:
model_config = {
    'embed_dim':    EMBED_DIM,
    'm_conditions': M_CONDITIONS,
    'text_dim':     TEXT_DIM,
    'img_size':     IMG_SIZE,
    'margin':       MARGIN,
    'lambda_l1':    LAMBDA_L1,
    'lambda_l2':    LAMBDA_L2,
    'backbone':     'resnet18_imagenet',
    'text_fusion':  'visual_text_elementwise',
    'framework':    'pytorch',
    'device':       str(DEVICE),
}

with open(OUT_DIR / 'model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

# Guardar pesos iniciales del modelo
torch.save(model.state_dict(), OUT_DIR / 'sce_net_init.pth')

print('Archivos guardados:')
print(f'  model_config.json  -> configuracion del modelo')
print(f'  sce_net_init.pth   -> pesos iniciales')
print()
print(json.dumps(model_config, indent=2))
print('\nSiguiente: 04_training.ipynb')